In [ ]:
!pip install scikit-image scikit-learn matplotlib seaborn opencv-python Pillow huggingface_hub pyyaml

import os
import json
import random
import numpy as np
import torch
import warnings

# 불필요한 시스템 경고 메시지를 숨겨 실행 결과창(Output)을 가독성 있게 유지합니다.
# 불필요한 경고 메시지가 많이 떠서 생성형 AI 활용하여 제거하는 코드 구성
warnings.filterwarnings('ignore')

print("=======================================================")
print("1. 전역 시드(Seed) 고정 및 Kaggle 인증 설정")
print("=======================================================")

# 재현성을 위한 전역 시드 고정 함수
# 모든 난수 생성기의 시드를 고정
def set_seed(seed=42):
    # 파이썬 내장 모듈 및 Numpy의 무작위성 제어
    random.seed(seed)
    np.random.seed(seed)

    # 파이썬 해시(Hash) 시드 고정
    os.environ['PYTHONHASHSEED'] = str(seed)

    # PyTorch의 연산 무작위성 제어
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


    # CuDNN 연산 무작위성 제거
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 시드값을 42로 통일하여 실행합니다.
set_seed(42)
print("Seed(42) 고정")


# 데이터 다운로드를 위한 Kaggle API 연동
# =====================================================================
# "username" : [kaggle 닉네임]
# "key" : [kaggle API KEY]
# =====================================================================
kaggle_credentials = {
    "username": "kaggle 닉네임",
    "key": "kaggle API 키"
}

# 인증 파일 자동 생성 및 권한 설정
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("Kaggle API 인증 완료")

1. 전역 시드(Seed) 고정 및 Kaggle 인증 설정
Seed(42) 고정
Kaggle API 인증 완료


In [ ]:
import shutil
import zipfile
from pathlib import Path
from huggingface_hub import snapshot_download

print("=======================================================")
print(" 2. 데이터셋 다운로드 및 압축 해제 ")
print("=======================================================")

# 다운로드 기본 경로 설정
# 경로를 변수화하여, 향후 경로 변경 시 유지보수 용이하도록 구성
DOWNLOAD_DIR = "/content/downloads"
skyborne_path = os.path.join(DOWNLOAD_DIR, "skyborne")
maryam_path = os.path.join(DOWNLOAD_DIR, "maryam")
seraphim_path = os.path.join(DOWNLOAD_DIR, "seraphim")

# Skyborne 데이터셋 다운로드
print("Skyborne Objects 다운로드")
!kaggle datasets download -d arminajdehnia/skyborne-objects-dataset -p {skyborne_path} --unzip

# Maryam 데이터셋 다운로드
print("Drone Detection 다운로드 중")
!kaggle datasets download -d maryamlsgumel/drone-detection-dataset -p {maryam_path} --unzip

# Seraphim 데이터셋 다운로드 (HuggingFace)
print("Seraphim Drone 다운로드 중")
snapshot_download(
    repo_id="lgrzybowski/seraphim-drone-detection-dataset",
    repo_type="dataset",
    local_dir=seraphim_path,
    local_dir_use_symlinks=False
)

# Seraphim 내부 중첩 ZIP 파일 압축 해제
print("Seraphim 내부 중첩 ZIP 파일 해제")
seraphim_zips = list(Path(seraphim_path).rglob('*.zip'))
for zip_file in seraphim_zips:
    extract_dir = zip_file.parent / zip_file.stem
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

print("모든 데이터셋 다운로드 및 압축 해제 완료")

 2. 데이터셋 다운로드 및 압축 해제 
Skyborne Objects 다운로드
Dataset URL: https://www.kaggle.com/datasets/arminajdehnia/skyborne-objects-dataset
License(s): CC-BY-NC-SA-4.0
100% 229M/229M [00:03<00:00, 78.0MB/s]

Drone Detection 다운로드 중
Dataset URL: https://www.kaggle.com/datasets/maryamlsgumel/drone-detection-dataset
License(s): CC-BY-NC-SA-4.0
100% 159M/159M [00:03<00:00, 53.0MB/s]

Seraphim Drone 다운로드 중


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

In [ ]:
import os
import shutil
from pathlib import Path

print("=======================================================")
print(" 3. 맞춤형 정밀 분류 및 데이터 통합 (raw_data) ")
print("=======================================================")

# 통합된 데이터를 저장할 경로 설정
TARGET_DIR = '/content/dataset/raw_data'

# 3가지 클래스 하위 폴더 생성
for cls in ['Drone', 'Airplane', 'Bird']:
    os.makedirs(os.path.join(TARGET_DIR, cls), exist_ok=True) # 폴더가 존재 시에도 에러 발생하지 않게 구성

# Seraphim 전용: 표준 YOLO 인덱스 기반 분류
def extract_seraphim(dataset_path, dataset_name):
    print(f"[{dataset_name}] 표준 YOLO 인덱스 기반 분류")
    img_files = [p for p in Path(dataset_path).rglob('*.*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    img_dict = {p.stem: p for p in img_files}
    label_files = [f for f in Path(dataset_path).rglob('*.txt') if f.name not in ['classes.txt', 'readme.txt', 'dataset.txt']]

    count = 0
    for label_file in label_files:
        try:
            with open(label_file, 'r') as f:
                lines = f.readlines()
        except: continue
        if lines and label_file.stem in img_dict:
            img_path = img_dict[label_file.stem]
            dest_path = os.path.join(TARGET_DIR, 'Drone', f"{dataset_name}_{img_path.name}")
            if not os.path.exists(dest_path):
                shutil.copy(str(img_path), dest_path)
                count += 1

    print(f"[Drone] {count:,}장 분류 및 이동 완료\n")

# Skyborne 전용: 텍스트 라벨 기반 분류
def extract_skyborne(dataset_path, dataset_name):
    print(f"[{dataset_name}] 텍스트 라벨 명세 기반 분류")
    img_files = [p for p in Path(dataset_path).rglob('*.*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    img_dict = {p.stem: p for p in img_files}
    label_files = [f for f in Path(dataset_path).rglob('*.txt') if f.name not in ['classes.txt', 'readme.txt', 'dataset.txt']]

    counts = {'Drone': 0, 'Airplane': 0, 'Bird': 0}
    for label_file in label_files:
        try:
            with open(label_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except: continue
        detected_targets = set()
        for line in lines:
            parts = line.strip().split()
            if parts:
                class_token = parts[0].lower()
                if 'drone' in class_token or 'uav' in class_token: detected_targets.add('Drone')
                elif any(x in class_token for x in ['airplane', 'aircraft', 'plane', 'helicopter', 'hellicopter']): detected_targets.add('Airplane')
                elif 'bird' in class_token: detected_targets.add('Bird')
        if detected_targets and label_file.stem in img_dict:
            img_path = img_dict[label_file.stem]
            for target in detected_targets:
                dest_path = os.path.join(TARGET_DIR, target, f"{dataset_name}_{img_path.name}")
                if not os.path.exists(dest_path):
                    shutil.copy(str(img_path), dest_path)
                    counts[target] += 1

    print(f"[Drone] {counts['Drone']:,}장 분류 및 이동 완료")
    print(f"[Airplane] {counts['Airplane']:,}장 분류 및 이동 완료")
    print(f"[Bird] {counts['Bird']:,}장 분류 및 이동 완료\n")

# Maryam 전용: 폴더명 기반 분류
def extract_maryam(dataset_path, dataset_name):
    print(f"[{dataset_name}] 직계 폴더명 정밀 매칭 기반 분류")
    img_files = [p for p in Path(dataset_path).rglob('*.*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

    counts = {'Drone': 0, 'Airplane': 0, 'Bird': 0}
    for img_path in img_files:
        parent_name = img_path.parent.name.lower()
        target_cls = None
        if 'drone' in parent_name: target_cls = 'Drone'
        elif any(x in parent_name for x in ['aeroplane', 'airplane', 'plane']): target_cls = 'Airplane'
        elif 'bird' in parent_name: target_cls = 'Bird'
        if target_cls:
            dest_path = os.path.join(TARGET_DIR, target_cls, f"{dataset_name}_{img_path.name}")
            if not os.path.exists(dest_path):
                shutil.copy(str(img_path), dest_path)
                counts[target_cls] += 1

    print(f"[Drone] {counts['Drone']:,}장 분류 및 이동 완료")
    print(f"[Airplane] {counts['Airplane']:,}장 분류 및 이동 완료")
    print(f"[Bird] {counts['Bird']:,}장 분류 및 이동 완료\n")

# 함수 실행 (앞서 설정해둔 경로 변수를 사용)
extract_seraphim(seraphim_path, "seraphim")
extract_skyborne(skyborne_path, "skyborne")
extract_maryam(maryam_path, "maryam")

# 최종 결과 요약 출력
print("=======================================================")
print("데이터셋 분류가 완벽하게 성공했습니다.")
print(f"최종 목적지: {TARGET_DIR}")
print(f"   - Airplane : {len(os.listdir(os.path.join(TARGET_DIR, 'Airplane'))):,}장")
print(f"   - Bird     : {len(os.listdir(os.path.join(TARGET_DIR, 'Bird'))):,}장")
print(f"   - Drone    : {len(os.listdir(os.path.join(TARGET_DIR, 'Drone'))):,}장")
print("=======================================================")

 3. 맞춤형 정밀 분류 및 데이터 통합 (raw_data) 
[seraphim] 표준 YOLO 인덱스 기반 분류
[Drone] 83,483장 분류 및 이동 완료

[skyborne] 텍스트 라벨 명세 기반 분류
[Drone] 630장 분류 및 이동 완료
[Airplane] 808장 분류 및 이동 완료
[Bird] 606장 분류 및 이동 완료

[maryam] 직계 폴더명 정밀 매칭 기반 분류
[Drone] 800장 분류 및 이동 완료
[Airplane] 1,010장 분류 및 이동 완료
[Bird] 582장 분류 및 이동 완료

데이터셋 분류가 완벽하게 성공했습니다.
최종 목적지: /content/dataset/raw_data
   - Airplane : 1,818장
   - Bird     : 1,188장
   - Drone    : 84,913장


In [ ]:
import cv2
import os
import random
import shutil
from pathlib import Path

print("=======================================================")
print("4. 결측치 제거 및 데이터 평형화 (Balancing)")
print("=======================================================")

# 경로 설정
CORRUPTED_DIR = '/content/dataset/corrupted_files'
BALANCED_DIR = '/content/dataset/balanced_data'
TARGET_DIR = '/content/dataset/raw_data'
classes = ['Drone', 'Airplane', 'Bird']
TARGET_COUNT = 900 # 클래스 평형화 개수

# 폴더 초기화 및 생성
os.makedirs(CORRUPTED_DIR, exist_ok=True)
if os.path.exists(BALANCED_DIR):
    shutil.rmtree(BALANCED_DIR)
for cls in classes:
    os.makedirs(os.path.join(BALANCED_DIR, cls), exist_ok=True)



# 1. 데이터 개수 및 결측치 추출 단계
print("--- 데이터 개수 및 결측치 추출 ---")

valid_files_dict = {cls: [] for cls in classes}
corrupted_count = 0

for cls in classes:
    cls_path = os.path.join(TARGET_DIR, cls)
    files = [f for f in Path(cls_path).rglob('*.*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    valid_count = 0

    for f in files:
        img = cv2.imread(str(f))
        # 결측치 판별
        if img is None:
            corrupted_count += 1
            new_name = f"{f.parent.name}_{f.name}"
            shutil.move(str(f), os.path.join(CORRUPTED_DIR, new_name))
        else:
            valid_files_dict[cls].append(f)
            valid_count += 1

    print(f"[{cls}] 유효한 이미지: {valid_count:,}장")

if corrupted_count > 0:
    print(f"\n 손상된 파일 {corrupted_count}장을 추출했습니다.")
    print(f"파일 저장 위치: {CORRUPTED_DIR}")
else:
    print("\n 결측치 분석 완료: 손상된 파일이 없습니다.")


print("\n")


# 2. 데이터 평형화(Balancing)
print("--- 데이터 평형화(Balancing) 진행 ---\n")

balanced_stats = {}

for cls in classes:
    valid_files = valid_files_dict[cls]

    # 시드 기반 무작위 900장 샘플링
    if len(valid_files) >= TARGET_COUNT:
        sampled = random.sample(valid_files, TARGET_COUNT) # 전역 시드 통제로 항상 동일한 사진 채택
        for f in sampled:
            shutil.copy(str(f), os.path.join(BALANCED_DIR, cls, f.name))
        balanced_stats[cls] = TARGET_COUNT
    else:
        for f in valid_files:
            shutil.copy(str(f), os.path.join(BALANCED_DIR, cls, f.name))
        balanced_stats[cls] = len(valid_files)

# 최종 결과 출력
print("==============================")
print(f"최종 목적지: {BALANCED_DIR}")
for cls in classes:
    print(f"- {cls} : {balanced_stats[cls]:,}장")
print("==============================")
print("클래스별 900장 평형화 완료")

4. 결측치 제거 및 데이터 평형화 (Balancing)
--- 데이터 개수 및 결측치 추출 ---
[Drone] 유효한 이미지: 84,879장
[Airplane] 유효한 이미지: 1,818장
[Bird] 유효한 이미지: 1,186장

 손상된 파일 34장을 추출했습니다.
파일 저장 위치: /content/dataset/corrupted_files


--- 데이터 평형화(Balancing) 진행 ---

최종 목적지: /content/dataset/balanced_data
- Drone : 900장
- Airplane : 900장
- Bird : 900장
클래스별 900장 평형화 완료


In [ ]:
from sklearn.model_selection import train_test_split

print("=======================================================")
print(" 5. 계층적 데이터 분할 (Stratified Split) ")
print("=======================================================")

# 실험용 파일 초기와 및 경로 설정
EXPERIMENT_DIR = '/content/dataset/experiment_data'
TRAIN_VAL_DIR = os.path.join(EXPERIMENT_DIR, 'train_val_set') # 학습 데이터셋 설정 80%
TEST_DIR = os.path.join(EXPERIMENT_DIR, 'test_set')           # 테스트 데이터셋 설정 20%

if os.path.exists(EXPERIMENT_DIR):
    shutil.rmtree(EXPERIMENT_DIR)
for cls in classes:
    os.makedirs(os.path.join(TRAIN_VAL_DIR, cls), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, cls), exist_ok=True)

# 전체 데이터 경로 및 라벨(정답) 리스트 설정
all_files, all_labels = [], []
for cls in classes:
    cls_path = os.path.join(BALANCED_DIR, cls)
    files = [str(f) for f in Path(cls_path).glob('*.*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    all_files.extend(files)
    all_labels.extend([cls] * len(files))

# 고정된 시드(42)로 항상 동일한 비율 분할
X_train_val, X_test, y_train_val, y_test = train_test_split(
    all_files, all_labels, test_size=0.20, random_state=42, stratify=all_labels
)

# 분할된 데이터를 폴더로 복사
for f, l in zip(X_train_val, y_train_val):
    shutil.copy(f, os.path.join(TRAIN_VAL_DIR, l, os.path.basename(f)))
for f, l in zip(X_test, y_test):
    shutil.copy(f, os.path.join(TEST_DIR, l, os.path.basename(f)))

print(f"데이터 세트 분할 완료: Train/Val({len(X_train_val)}장) | Test({len(X_test)}장)")

 5. 계층적 데이터 분할 (Stratified Split) 
데이터 세트 분할 완료: Train/Val(2160장) | Test(540장)


In [ ]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

print("=======================================================")
print(" 6. 머신러닝: HOG 특징 벡터 추출 파이프라인 ")
print("=======================================================")


# 경로 및 HOG 추출 환경 설정
EXPERIMENT_DIR = '/content/dataset/experiment_data'
TRAIN_VAL_DIR = os.path.join(EXPERIMENT_DIR, 'train_val_set')
TEST_DIR = os.path.join(EXPERIMENT_DIR, 'test_set')

# HOG 파라미터 및 이미지 리사이즈(정규화)
IMG_SIZE = (128, 128)

def extract_hog_features(data_dir):
    features = []
    labels = []
    classes = ['Drone', 'Airplane', 'Bird']

    for cls in classes:
        cls_path = os.path.join(data_dir, cls)
        files = [f for f in Path(cls_path).glob('*.*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]

        print(f"[{cls}] 데이터 HOG 추출 (총 {len(files)}장)")
        for f in files:
            img = cv2.imread(str(f))
            if img is not None:
                # 1. Grayscale 변환 및 크기 정규화
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                resized = cv2.resize(gray, IMG_SIZE)

                # 2. HOG 특징 추출 - 보고서에 작성해둔 방법론 활용
                fd = hog(resized, orientations=9, pixels_per_cell=(8, 8),
                         cells_per_block=(2, 2), visualize=False)

                features.append(fd)
                labels.append(cls)

    return np.array(features), np.array(labels)

print("\n[학습/검증 데이터(Train/Val) HOG 추출 시작]")
X_train_val_hog, y_train_val_str = extract_hog_features(TRAIN_VAL_DIR)

print("\n[테스트 데이터(Test) HOG 추출 시작]")
X_test_hog, y_test_str = extract_hog_features(TEST_DIR)

# 라벨 인코딩 (문자열을 숫자로 변경 : Airplane:0, Bird:1, Drone:2)
le = LabelEncoder()
y_train_val_hog = le.fit_transform(y_train_val_str)
y_test_hog = le.transform(y_test_str)

print(f"\nHOG 추출 완료! 특징 벡터 차원 크기: {X_train_val_hog.shape[1]}")

 6. 머신러닝: HOG 특징 벡터 추출 파이프라인 

[학습/검증 데이터(Train/Val) HOG 추출 시작]
[Drone] 데이터 HOG 추출 (총 720장)
[Airplane] 데이터 HOG 추출 (총 720장)
[Bird] 데이터 HOG 추출 (총 720장)

[테스트 데이터(Test) HOG 추출 시작]
[Drone] 데이터 HOG 추출 (총 180장)
[Airplane] 데이터 HOG 추출 (총 180장)
[Bird] 데이터 HOG 추출 (총 180장)

HOG 추출 완료! 특징 벡터 차원 크기: 8100


In [ ]:
import joblib
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("=======================================================")
print(" 7. SVM 최적화, 평가 및 모델 파일 저장 ")
print("=======================================================")

scaler = StandardScaler()
X_train_val_scaled = scaler.fit_transform(X_train_val_hog)
X_test_scaled = scaler.transform(X_test_hog)

param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [1, 0.1, 0.01, 0.001], 'kernel': ['rbf']}

# 모델 난수(42) 고정
svm = SVC(random_state=42)
grid_search = GridSearchCV(svm, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)

print("▶ 5-Fold GridSearchCV 튜닝 중... (수 분 소요)")
grid_search.fit(X_train_val_scaled, y_train_val_hog)

# 가장 성능이 높았던 모델 가져옴
best_svm = grid_search.best_estimator_

# 테스트 데이터로 최종 예측 수행
y_pred = best_svm.predict(X_test_scaled)
print(f"최적 파라미터: {grid_search.best_params_}")
print("\n[베이스라인 모델(HOG+SVM) 테스트 결과]")
print(f" - Accuracy: {accuracy_score(y_test_hog, y_pred) * 100:.2f}%")
print(f" - F1-Score (Macro): {f1_score(y_test_hog, y_pred, average='macro'):.4f}")
print("\n[상세 분류 결과]")
print(classification_report(y_test_hog, y_pred, target_names=le.classes_))

 7. SVM 최적화, 평가 및 모델 파일 저장 
▶ 5-Fold GridSearchCV 튜닝 중... (수 분 소요)
최적 파라미터: {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}

[베이스라인 모델(HOG+SVM) 테스트 결과]
 - Accuracy: 52.41%
 - F1-Score (Macro): 0.4703

[상세 분류 결과]
              precision    recall  f1-score   support

    Airplane       1.00      0.11      0.19       180
        Bird       1.00      0.47      0.64       180
       Drone       0.41      1.00      0.58       180

    accuracy                           0.52       540
   macro avg       0.80      0.52      0.47       540
weighted avg       0.80      0.52      0.47       540



In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

print("=======================================================")
print(" 8. 제안 모델: ResNet-18 데이터 파이프라인 구성 ")
print("=======================================================")

# 연산 장치(Device) 할당
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 데이터 증강 및 텐서 변환
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val_test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 데이터셋 생성 및 학습/검증 데이터셋 분할
full_train_val_dataset = datasets.ImageFolder(TRAIN_VAL_DIR, transform=data_transforms['train'])
test_dataset = datasets.ImageFolder(TEST_DIR, transform=data_transforms['val_test'])

train_size = int(0.8 * len(full_train_val_dataset))
val_size = len(full_train_val_dataset) - train_size

# 고정된 랜덤 시드 활용하여 제너레이터 분할
gen = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_train_val_dataset, [train_size, val_size], generator=gen)
val_dataset.dataset.transform = data_transforms['val_test']

BATCH_SIZE = 32


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, worker_init_fn=lambda worker_id: np.random.seed(42 + worker_id))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = full_train_val_dataset.classes

print(f"클래스 목록: {class_names}")
print("데이터 로더 생성 완료")

 8. 제안 모델: ResNet-18 데이터 파이프라인 구성 
클래스 목록: ['Airplane', 'Bird', 'Drone']
데이터 로더 생성 완료


In [ ]:
import copy
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch

print("=======================================================")
print(" 9. ResNet-18 전이 학습, 조기 종료 및 가중치 평가 ")
print("=======================================================")

model = models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()

# 하이퍼파라미터 설정
LEARNING_RATE = 1e-4
OPTIMIZER_NAME = 'Adam'
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

EPOCHS = 50
PATIENCE = 7

best_model_wts = copy.deepcopy(model.state_dict())
best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0

train_losses, val_losses, train_accs, val_accs = [], [], [], []
print("모델 훈련 시작")

for epoch in range(EPOCHS):
    model.train()
    running_loss, running_corrects = 0.0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_acc = running_corrects.double() / len(train_dataset)
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc.item())

    model.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = correct.double() / len(val_dataset)
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc.item())

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

    # 최적 에포크 기록
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n[조기 종료]")
            break


print("\n모델 학습 완료")
print(f"최적 파라미터: {{'Optimizer': '{OPTIMIZER_NAME}', 'Learning Rate': {LEARNING_RATE}, 'Batch Size': {BATCH_SIZE}}}")
print(f"최적 가중치: [Epoch {best_epoch}] 시점의 가중치 (Best Val Loss: {best_val_loss:.4f})")


# 최종 가중치 로드 및 평가
model.load_state_dict(best_model_wts)
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

resnet_acc = accuracy_score(all_labels, all_preds)
resnet_f1 = f1_score(all_labels, all_preds, average='macro')

print("\n [제안 모델(ResNet-18) 최종 테스트 결과]")
print(f" - 정확도 (Accuracy): {resnet_acc * 100:.2f}%")
print(f" - F1-Score (Macro): {resnet_f1:.4f}\n")

print("\n[상세 분류 결과]")
print(classification_report(all_labels, all_preds, target_names=class_names))

 9. ResNet-18 전이 학습, 조기 종료 및 가중치 평가 
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 165MB/s]


모델 훈련 시작
Epoch 01/50 | Train Loss: 0.3659 | Val Loss: 0.1982
Epoch 02/50 | Train Loss: 0.0699 | Val Loss: 0.1914
Epoch 03/50 | Train Loss: 0.0256 | Val Loss: 0.1710
Epoch 04/50 | Train Loss: 0.0163 | Val Loss: 0.1950
Epoch 05/50 | Train Loss: 0.0141 | Val Loss: 0.2427
Epoch 06/50 | Train Loss: 0.0075 | Val Loss: 0.1980
Epoch 07/50 | Train Loss: 0.0047 | Val Loss: 0.1901
Epoch 08/50 | Train Loss: 0.0028 | Val Loss: 0.2172
Epoch 09/50 | Train Loss: 0.0032 | Val Loss: 0.1896
Epoch 10/50 | Train Loss: 0.0028 | Val Loss: 0.2157

[조기 종료]

모델 학습 완료
최적 파라미터: {'Optimizer': 'Adam', 'Learning Rate': 0.0001, 'Batch Size': 32}
최적 가중치: [Epoch 3] 시점의 가중치 (Best Val Loss: 0.1710)

 [제안 모델(ResNet-18) 최종 테스트 결과]
 - 정확도 (Accuracy): 94.44%
 - F1-Score (Macro): 0.9446


[상세 분류 결과]
              precision    recall  f1-score   support

    Airplane       0.90      0.97      0.93       180
        Bird       0.95      0.96      0.96       180
       Drone       0.99      0.91      0.94       180

    accuracy